# Day 3 — Llama RAG Assistant

## Insurellm Expert Knowledge Worker

This notebook combines the **data preparation/vector-building process from Day 2** with a stronger free/local RAG pipeline.

The notebook is designed to run from a fresh Colab runtime:

**knowledge-base.zip → documents → chunks → embeddings → Chroma → retrieval → reranking → Llama-family generation → Gradio**

The generation model is a small Llama-family chat model, so we do not depend on OpenAI or another paid API.

In [1]:
# ============================================================
# STEP 1 — Install required packages
# ============================================================
# These packages provide document loading, chunking, embeddings,
# Chroma vector storage, local Hugging Face generation, and Gradio.

!pip install -q -U \
    langchain \
    langchain-chroma \
    langchain-huggingface \
    langchain-community \
    langchain-text-splitters \
    sentence-transformers \
    transformers \
    accelerate \
    gradio


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/7

In [2]:
# ============================================================
# STEP 2 — Import libraries
# ============================================================

import os
import glob
import zipfile
import math
import torch

from pathlib import Path

from langchain_chroma import Chroma
from langchain_huggingface import (
    HuggingFaceEmbeddings,
    HuggingFacePipeline,
)
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline,
)

import gradio as gr

print("All imports completed successfully.")


/tmp/ipykernel_659/3242419167.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


All imports completed successfully.


## STEP 3 — Upload the knowledge base

Day 2 used `knowledge-base.zip`. We use the same source here.

In a fresh Google Colab runtime, run this cell and upload **`knowledge-base.zip`**.


In [3]:
# ============================================================
# STEP 3 — Upload knowledge-base.zip
# ============================================================

from google.colab import files

KNOWLEDGE_ZIP = "knowledge-base.zip"

if Path(KNOWLEDGE_ZIP).exists():
    print(f"{KNOWLEDGE_ZIP} is already available.")
else:
    print(f"Please upload {KNOWLEDGE_ZIP}.")
    uploaded = files.upload()

    if KNOWLEDGE_ZIP not in uploaded and not Path(KNOWLEDGE_ZIP).exists():
        raise FileNotFoundError(
            f"{KNOWLEDGE_ZIP} was not uploaded. "
            "Rerun this cell and upload the correct file."
        )

print("Knowledge-base ZIP is ready.")


Please upload knowledge-base.zip.


Saving knowledge-base.zip to knowledge-base.zip
Knowledge-base ZIP is ready.


In [4]:
# ============================================================
# STEP 4 — Extract the knowledge base
# ============================================================

KNOWLEDGE_BASE_DIR = Path("knowledge-base")

if not KNOWLEDGE_BASE_DIR.exists():
    with zipfile.ZipFile(KNOWLEDGE_ZIP, "r") as zip_ref:
        zip_ref.extractall(".")

    print("Knowledge base extracted successfully.")
else:
    print("Knowledge base folder already exists. Using it.")

markdown_files = glob.glob(
    "knowledge-base/**/*.md",
    recursive=True,
)

print(f"Found {len(markdown_files)} Markdown files.")

if not markdown_files:
    raise FileNotFoundError(
        "No Markdown files were found inside knowledge-base/. "
        "Check the ZIP structure."
    )


Knowledge base extracted successfully.
Found 76 Markdown files.


In [5]:
# ============================================================
# STEP 5 — Inspect the knowledge base
# ============================================================
# This is a sanity check. We later split the text into smaller
# chunks before creating embeddings.

entire_knowledge_base = ""

for file_path in markdown_files:
    with open(file_path, "r", encoding="utf-8") as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")


Total characters in knowledge base: 304,434


## STEP 6 — Load all Markdown documents

Each Markdown file becomes a LangChain `Document`.

We also store the folder name as `doc_type`, which helps us understand where retrieved information came from.


In [6]:
# ============================================================
# STEP 6 — Load all Markdown documents
# ============================================================

folders = glob.glob("knowledge-base/*")

documents = []

for folder in folders:
    # Example values: company, employees, products, contracts.
    doc_type = os.path.basename(folder)

    loader = DirectoryLoader(
        folder,
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )

    folder_docs = loader.load()

    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents.")

if not documents:
    raise ValueError("No documents were loaded.")


Loaded 76 documents.


In [7]:
# ============================================================
# STEP 7 — Split documents into chunks
# ============================================================
# Smaller chunks make semantic search more precise.
#
# chunk_size = 1000 characters
# chunk_overlap = 200 characters
#
# The overlap helps preserve information across chunk boundaries.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = text_splitter.split_documents(documents)

print(f"Divided the knowledge base into {len(chunks)} chunks.")

if not chunks:
    raise ValueError("Chunking produced zero chunks.")

print("\nExample first chunk:")
print(chunks[0].page_content[:1000])

print("\nExample metadata:")
print(chunks[0].metadata)


Divided the knowledge base into 413 chunks.

Example first chunk:
# HR Record

# Jessica Liu

## Summary
- **Date of Birth:** April 30, 1996
- **Job Title:** Frontend Developer
- **Location:** Remote (Based in Seattle, Washington)
- **Current Salary:** $92,000

## Insurellm Career Progression
- **July 2022 - Present:** Frontend Developer
  - Develops user interfaces for Rellm reinsurance platform using React
  - Implements responsive designs and ensures cross-browser compatibility
  - Collaborates with UX designers and backend engineers

- **January 2020 - June 2022:** Junior Frontend Developer
  - Built UI components for internal tools and customer-facing applications
  - Fixed bugs and improved performance of existing web applications
  - Participated in code reviews and learned best practices

- **June 2018 - December 2019:** Web Developer Intern at StartupLabs
  - Created landing pages and marketing websites
  - Learned HTML, CSS, JavaScript, and React fundamentals

Example metadat

## STEP 8 — Create the free embedding model

`all-MiniLM-L6-v2` converts each chunk into a 384-dimensional vector.

The same embedding model is used for questions, so Chroma can compare the question vector with the stored document vectors.


In [8]:
# ============================================================
# STEP 8 — Create the free embedding model
# ============================================================

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={
        "normalize_embeddings": True,
    },
)

print("Embedding model loaded successfully.")
print("Model:", EMBEDDING_MODEL)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.
Model: sentence-transformers/all-MiniLM-L6-v2


## STEP 9 — Build and populate Chroma

This is the important fix for the previous `Documents/vectors: 0` problem.

Instead of assuming that a previous Chroma database exists, we create the vector store directly from the chunks.

If an old collection exists, it is deleted first so rerunning the notebook does not duplicate all chunks.


In [9]:
# ============================================================
# STEP 9 — Create and populate Chroma
# ============================================================

DB_NAME = "vector_db"

# Remove an old collection so rerunning this notebook starts clean.
if Path(DB_NAME).exists():
    try:
        old_vectorstore = Chroma(
            persist_directory=DB_NAME,
            embedding_function=embeddings,
        )
        old_vectorstore.delete_collection()
        print("Old Chroma collection deleted.")
    except Exception as e:
        print("No usable old Chroma collection was found.")
        print("Details:", e)

# Create the new Chroma collection and embed every chunk.
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME,
)

# Verify that vectors were actually inserted.
vector_count = vectorstore._collection.count()

print("Chroma vector store created successfully.")
print(f"Vectors/documents stored: {vector_count}")

if vector_count == 0:
    raise RuntimeError(
        "Chroma contains 0 vectors. "
        "The embedding/storage step did not succeed."
    )


Chroma vector store created successfully.
Vectors/documents stored: 413


In [10]:
# ============================================================
# STEP 10 — Verify vector dimensions
# ============================================================
# This confirms that Chroma contains actual embeddings.
#
# IMPORTANT:
# sample["embeddings"] is a NumPy array, so we must check
# whether it is None or whether its length is zero.
# We should NOT write:
#
#     if not sample["embeddings"]:
#
# because NumPy arrays cannot be evaluated that way.

collection = vectorstore._collection

sample = collection.get(
    limit=1,
    include=["embeddings"],
)

# Check that the embeddings field exists.
if sample.get("embeddings") is None:
    raise RuntimeError(
        "Chroma returned no embeddings."
    )

# Convert to a normal Python/NumPy object and check its size.
embeddings_array = sample["embeddings"]

if len(embeddings_array) == 0:
    raise RuntimeError(
        "Chroma contains no embedding vectors."
    )

# Get the first stored vector.
first_embedding = embeddings_array[0]

# Count the dimensions of that vector.
dimensions = len(first_embedding)

print(
    f"There are {vector_count:,} vectors "
    f"with {dimensions:,} dimensions in Chroma."
)

# all-MiniLM-L6-v2 normally produces 384-dimensional vectors.
if dimensions != 384:
    print(
        f"Warning: expected 384 dimensions, "
        f"but found {dimensions}."
    )
else:
    print("Embedding dimension check passed: 384.")

There are 413 vectors with 384 dimensions in Chroma.
Embedding dimension check passed: 384.


## STEP 11 — Create the retriever

Normal RAG questions retrieve 10 candidate chunks.

For debugging, the underlying Chroma collection can also be queried directly in the next step.


In [11]:
# ============================================================
# STEP 11 — Create the retriever
# ============================================================

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10},
)

print("Retriever created successfully.")


Retriever created successfully.


## STEP 12 — Test retrieval before generation

Always test retrieval separately first.

If the correct document is not retrieved, changing the language model will not solve the problem.


In [12]:
# ============================================================
# STEP 12 — Test retrieval
# ============================================================

test_question = "Who won the prestigious IIOTY award in 2023?"

retrieved_docs = retriever.invoke(test_question)

print("Question:")
print(test_question)

print(f"\nNumber of retrieved documents: {len(retrieved_docs)}")

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Retrieved document {i} ---")
    print("TYPE:", doc.metadata.get("doc_type"))
    print("SOURCE:", doc.metadata.get("source"))
    print("CONTENT:")
    print(doc.page_content[:1000])


Question:
Who won the prestigious IIOTY award in 2023?

Number of retrieved documents: 10

--- Retrieved document 1 ---
TYPE: employees
SOURCE: knowledge-base/employees/Oliver Spencer.md
CONTENT:
## Annual Performance History
- **2018**: **3/5** - Adaptable team player but still learning to take initiative.
- **2019**: **4/5** - Demonstrated strong problem-solving skills, outstanding contribution on the claims project.
- **2020**: **2/5** - Struggled with time management; fell behind on deadlines during a high-traffic release period.
- **2021**: **4/5** - Made a significant turnaround with organized work habits and successful project management.
- **2022**: **5/5** - Exceptional performance during the "Innovate" initiative, showcasing leadership and creativity.
- **2023**: **3/5** - Maintaining steady work; expectations for innovation not fully met, leading to discussions about goals.

--- Retrieved document 2 ---
TYPE: employees
SOURCE: knowledge-base/employees/Alex Chen.md
CONTENT:
#

In [13]:
# ============================================================
# STEP 13 — Direct Chroma debugging query
# ============================================================
# This cell lets us inspect exactly what Chroma returns.
# It is especially useful if retrieval results look wrong.

query = "Who won the prestigious IIOTY award in 2023?"

query_embedding = embeddings.embed_query(query)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10,
    include=["documents", "metadatas", "distances"],
)

print("Direct Chroma query:")
print(query)

for i, doc in enumerate(results["documents"][0], start=1):
    metadata = results["metadatas"][0][i - 1]
    distance = results["distances"][0][i - 1]

    print(f"\n--- Result {i} ---")
    print("Distance:", distance)
    print("Metadata:", metadata)
    print("Content:")
    print(doc[:1000])


Direct Chroma query:
Who won the prestigious IIOTY award in 2023?

--- Result 1 ---
Distance: 1.1774286031723022
Metadata: {'doc_type': 'employees', 'source': 'knowledge-base/employees/Oliver Spencer.md'}
Content:
## Annual Performance History
- **2018**: **3/5** - Adaptable team player but still learning to take initiative.
- **2019**: **4/5** - Demonstrated strong problem-solving skills, outstanding contribution on the claims project.
- **2020**: **2/5** - Struggled with time management; fell behind on deadlines during a high-traffic release period.
- **2021**: **4/5** - Made a significant turnaround with organized work habits and successful project management.
- **2022**: **5/5** - Exceptional performance during the "Innovate" initiative, showcasing leadership and creativity.
- **2023**: **3/5** - Maintaining steady work; expectations for innovation not fully met, leading to discussions about goals.

--- Result 2 ---
Distance: 1.1794261932373047
Metadata: {'doc_type': 'employees', '

## STEP 14 — Load a free Llama-family language model

We now add the generation part of RAG.

This version uses **TinyLlama-1.1B-Chat-v1.0**, a small Llama-family instruct/chat model that can run locally in Colab without a paid API.

Why TinyLlama here?
- It is free to download and run.
- It does not require an API key.
- It works with the normal Transformers `text-generation` pipeline.
- It avoids the `text2text-generation` compatibility problem we had with FLAN-T5.

The notebook also includes a stronger optional Llama model setting below.

In [14]:
# ============================================================
# STEP 14A — Select GPU or CPU
# ============================================================

# A GPU makes local Llama generation much faster.
USE_CUDA = torch.cuda.is_available()

if USE_CUDA:
    print("GPU detected. Llama generation will use the GPU.")
else:
    print("No GPU detected. Llama generation will use the CPU.")

print("CUDA available:", USE_CUDA)

GPU detected. Llama generation will use the GPU.
CUDA available: True


In [15]:
# ============================================================
# STEP 14B — Load a free Llama-family model
# ============================================================

# TinyLlama is a Llama-family chat model small enough for a
# normal free Colab environment.
# We deliberately use direct Transformers generation instead
# of the text2text-generation pipeline.

GENERATION_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL)

if USE_CUDA:
    model = AutoModelForCausalLM.from_pretrained(
        GENERATION_MODEL,
        torch_dtype=torch.float16,
    ).to("cuda")
else:
    model = AutoModelForCausalLM.from_pretrained(
        GENERATION_MODEL
    ).to("cpu")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Llama-family model loaded successfully.")
print("Model:", GENERATION_MODEL)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Llama-family model loaded successfully.
Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0


## STEP 15 — Define the grounded RAG prompt

The language model receives the retrieved context and the user's question.

The instructions tell it to use only the supplied documents and not invent facts.


In [16]:
# ============================================================
# STEP 15 — Define a grounded RAG prompt
# ============================================================

# The prompt is intentionally strict because a small local model
# should not be allowed to invent facts or mix unrelated employees.

RAG_PROMPT = '''You are the Insurellm knowledge-base assistant.

Answer the user's question using ONLY the CONTEXT below.

Rules:
- Give a direct answer.
- Do not invent facts.
- Do not combine information from unrelated people.
- If the context does not contain the answer, say:
  "I don't have enough information in the knowledge base."
- For person and award questions, use the exact name and award
  information supported by the context.
- Keep the answer to 1-3 sentences.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
'''

In [17]:
# ============================================================
# STEP 16 — Build the complete RAG function
# ============================================================

import re

def _tokenize_for_matching(text):
    """Return simple lowercase word tokens for lexical matching."""
    return set(re.findall(r"\b[a-zA-Z0-9][a-zA-Z0-9_-]*\b", text.lower()))


def rerank_documents(question, documents, top_k=4):
    """Combine semantic retrieval with lightweight lexical reranking."""
    question_tokens = _tokenize_for_matching(question)
    stop_words = {
        "who", "what", "when", "where", "why", "how", "is", "was",
        "were", "the", "a", "an", "in", "on", "of", "for", "to",
        "did", "do", "does", "and", "or", "with", "about", "at"
    }
    important_tokens = {t for t in question_tokens if t not in stop_words}
    scored = []

    for original_rank, doc in enumerate(documents, start=1):
        doc_tokens = _tokenize_for_matching(doc.page_content)
        overlap = important_tokens.intersection(doc_tokens)
        lexical_score = len(overlap)
        rank_bonus = 1.0 / original_rank
        final_score = lexical_score * 3.0 + rank_bonus
        scored.append((final_score, original_rank, doc))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [item[2] for item in scored[:top_k]]


def build_context(documents, max_chars_per_doc=1800):
    """Build a compact context so unrelated text does not overwhelm the model."""
    parts = []
    for index, doc in enumerate(documents, start=1):
        text = doc.page_content.strip()[:max_chars_per_doc]
        source = doc.metadata.get("source", "unknown")
        doc_type = doc.metadata.get("doc_type", "unknown")
        parts.append(
            f"[SOURCE {index} | TYPE: {doc_type} | FILE: {source}]\n{text}"
        )
    return "\n\n---\n\n".join(parts)


def answer_question(question: str, history=None):
    """Run retrieval, reranking, grounded Llama generation, and validation."""

    # 1. Retrieve semantic candidates from Chroma.
    candidates = retriever.invoke(question)

    if not candidates:
        return "I don't have enough information in the knowledge base."

    # 2. Rerank using important question terms.
    ranked_docs = rerank_documents(question, candidates, top_k=4)

    # 3. Keep only focused context.
    context = build_context(ranked_docs, max_chars_per_doc=1800)

    # 4. Build the grounded prompt.
    prompt = RAG_PROMPT.format(context=context, question=question)

    # 5. Tokenize.
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    )

    if USE_CUDA:
        inputs = {key: value.to("cuda") for key, value in inputs.items()}

    # 6. Generate deterministically and discourage repetition.
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # 7. Decode only newly generated tokens, not the prompt.
    input_length = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, input_length:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    # 8. Remove occasional template markers.
    for marker in ["### Assistant:", "Assistant:", "ANSWER:"]:
        if marker in answer:
            answer = answer.split(marker, 1)[-1].strip()

    # 9. If the small model returns an empty or highly repetitive result,
    # return the strongest evidence instead of nonsense.
    words = answer.split()
    too_short = len(words) == 0
    repetitive = len(words) >= 8 and len(set(words)) / len(words) < 0.45

    if too_short or repetitive:
        return ranked_docs[0].page_content.strip()[:1200]

    return answer

## STEP 17 — Test the IIOTY question

This checks the complete pipeline:

**Question → embedding → Chroma → retrieved context → FLAN-T5 → answer**


In [18]:
# ============================================================
# STEP 17 — Test the IIOTY question
# ============================================================

question = "Who won the prestigious IIOTY award in 2023?"
answer = answer_question(question)

print("Question:")
print(question)
print("\nAnswer:")
print(answer)

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question:
Who won the prestigious IIOTY award in 2023?

Answer:
The InsureLLM Innovators of the Year award went to Maxine Thompson, Senior Data Analyst, for her outstanding contribution to the company' s innovation efforts.


In [19]:
# ============================================================
# STEP 18 — Test several knowledge-base questions
# ============================================================

test_questions = [
    "Who is Maxine Thompson?",
    "Who won the prestigious IIOTY award in 2023?",
    "What was Maxine's role at Insurellm?",
    "When did Maxine become a Senior Data Engineer?",
    "What was the Innovate initiative?",
]

for question in test_questions:
    print("\n" + "=" * 70)
    print("QUESTION:")
    print(question)
    print("\nANSWER:")
    print(answer_question(question))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION:
Who is Maxine Thompson?

ANSWER:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Maxine is an experienced data engineer who has been working at Insurellm since 2 years ago. She has received multiple awards for her work, including the Insultec Young Talent Award in the same year she joined the company.

QUESTION:
Who won the prestigious IIOTY award in 2023?

ANSWER:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The InsureLLM Innovators of the Year award went to Maxine Thompson, Senior Data Analyst, for her outstanding contribution to the company' s innovation efforts.

QUESTION:
What was Maxine's role at Insurellm?

ANSWER:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


She was promoted from a Junior Analyst to a Senior Data Analyst before being appointed as a Data Engineeer.

QUESTION:
When did Maxine become a Senior Data Engineer?

ANSWER:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


InsurellM Innovators of the Year 21

QUESTION:
What was the Innovate initiative?

ANSWER:
The Innovat initiative aimed to create a new product line focused on providing innovative solutions to insurance companies.


## STEP 19 — Launch the Gradio chat interface

The Gradio interface uses the same retrieval, reranking, and Llama generation function tested above.

Everything remains local/free: Chroma, embeddings, reranking, and generation.

In [ ]:
# ============================================================
# STEP 19 — Start the Gradio application
# ============================================================

demo = gr.ChatInterface(
    fn=answer_question,
    title="Insurellm RAG Assistant",
    description=(
        "Ask questions about the Insurellm knowledge base. "
        "Answers are generated from the documents stored in Chroma."
    ),
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://90b2fe39059b4f46e7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


# Final Summary

The updated Day 3 pipeline is:

**knowledge-base.zip → extract → load Markdown → split → MiniLM embeddings → Chroma → semantic retrieval → lexical/semantic reranking → focused context → Llama-family generation → validation fallback → Gradio**

### Model

Default generation model: `TinyLlama/TinyLlama-1.1B-Chat-v1.0`

It is selected because it is a small Llama-family chat model that can run locally without a paid API or Hugging Face access token.

### Important verification

After Step 9, Chroma must report a non-zero vector count.
After Step 10, the MiniLM embeddings should have 384 dimensions.
After Step 17, inspect the IIOTY result before launching Gradio.